# Target Encoding Experiment

This notebook tests smoothed target encoding as an alternative to one-hot encoding for categorical variables in the subrogation prediction project.

The goal is to compare these models against the one-hot encoded baselines from `04_model_baseline.ipynb`:

- `logistic_regression_target_encoding_v1`
- `xgboost_target_encoding_v1`

Target encoding is learned from the training split only to avoid validation leakage.

## 1. Imports and project paths

In [21]:
import json
import joblib
import numpy as np
import pandas as pd

from pathlib import Path
from datetime import datetime

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

from xgboost import XGBClassifier

In [22]:
# If this notebook is inside a notebooks/ folder, this points to the project root.
# Otherwise, it uses the current working directory as the project root.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "train_features.csv"
MODELS_DIR = PROJECT_ROOT / "models"
REPORTS_DIR = PROJECT_ROOT / "reports"
METRICS_PATH = REPORTS_DIR / "model_metrics.json"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data path:", DATA_PATH)
print("Metrics path:", METRICS_PATH)

Project root: /Users/eugene/Desktop/Emory/Projects/subrogation-project
Data path: /Users/eugene/Desktop/Emory/Projects/subrogation-project/data/processed/train_features.csv
Metrics path: /Users/eugene/Desktop/Emory/Projects/subrogation-project/reports/model_metrics.json


## 2. Load engineered training data

This notebook assumes `03_feature_engineering.ipynb` has already created `data/processed/train_features.csv`.

In [23]:
df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
df.head()

Shape: (17999, 42)


,subrogation,email_or_tel_available,safety_rating,high_education_ind,address_change_ind,accident_site,witness_present_ind,liab_prct,channel,policy_report_filed_ind,...,high_value_low_liability_ind,high_value_strong_docs_ind,payout_vehicle_ratio_band,vehicle_value_band,vehicle_age_at_claim,mileage_per_vehicle_year,in_network_bodyshop_flag,in_network_high_value_claim_ind,weekend_claim_ind,accident_type_site
0,1,0,75.0,1,1,Parking Area,1,31.0,Broker,1,...,1,1,low,"(14999.999, 19639.785]",0.0,75421.0,0,0,1,multi_vehicle_clear_Parking Area
1,0,1,94.0,1,1,Parking Area,0,34.0,Phone,1,...,0,0,very_low,"(19639.785, 42609.417]",0.0,31988.0,1,0,0,multi_vehicle_clear_Parking Area
2,0,1,76.0,1,1,Unknown,0,39.0,Broker,1,...,1,1,low,"(14999.999, 19639.785]",0.0,60876.0,1,1,0,multi_vehicle_unclear_Unknown
3,1,1,54.0,1,1,Unknown,0,32.0,Phone,1,...,0,0,very_low,"(14999.999, 19639.785]",0.0,152772.0,1,0,1,multi_vehicle_unclear_Unknown
4,0,1,54.0,1,1,Highway/Intersection,0,28.0,Online,0,...,1,0,low,"(42609.417, 130000.0]",0.0,41151.0,1,1,1,multi_vehicle_clear_Highway/Intersection


In [24]:
# Basic target check
print(df["subrogation"].value_counts(normalize=True).rename("proportion"))
print("Missing target values:", df["subrogation"].isna().sum())

subrogation
0    0.771376
1    0.228624
Name: proportion, dtype: float64
Missing target values: 0


## 3. Define features and target

We drop identifier columns because they should not be used as predictive features.

In [25]:
TARGET_COL = "subrogation"
ID_COLS = ["claim_number"]

X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

# Drop identifier columns if present
X = X.drop(columns=[col for col in ID_COLS if col in X.columns])

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (17999, 41)
y shape: (17999,)


In [26]:
# Identify categorical columns for target encoding
categorical_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
numeric_cols = X.select_dtypes(exclude=["object", "category"]).columns.tolist()

print("Categorical columns:", categorical_cols)
print("Number of categorical columns:", len(categorical_cols))
print("Number of numeric columns:", len(numeric_cols))

Categorical columns: ['accident_site', 'channel', 'accident_type', 'in_network_bodyshop', 'liability_band', 'claim_amount_band', 'payout_vehicle_ratio_band', 'vehicle_value_band', 'accident_type_site']
Number of categorical columns: 9
Number of numeric columns: 32


## 4. Train/validation split

The split happens before target encoding so validation target values are never used to create encoded features.

In [27]:
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("y_train positive rate:", round(y_train.mean(), 4))
print("y_val positive rate:", round(y_val.mean(), 4))

X_train: (14399, 41)
X_val: (3600, 41)
y_train positive rate: 0.2286
y_val positive rate: 0.2286


## 5. Smoothed target encoding functions

For validation data, each category is encoded using target statistics from the training split only.

For training data, this notebook uses out-of-fold target encoding. That prevents each training row from being encoded using its own target value, which reduces overfitting.

In [28]:
def _smoothed_category_mapping(X_col, y, smoothing=20):
    """
    Create a smoothed category-to-target mapping for one column.

    Smoothed mean = (count * category_mean + smoothing * global_mean) / (count + smoothing)
    """
    global_mean = y.mean()

    stats = (
        pd.DataFrame({"category": X_col.astype(str), "target": y})
        .groupby("category")["target"]
        .agg(["mean", "count"])
    )

    smoothed_mean = (
        (stats["count"] * stats["mean"] + smoothing * global_mean)
        / (stats["count"] + smoothing)
    )

    return smoothed_mean.to_dict(), global_mean


def target_encode_train_validation(
    X_train,
    X_val,
    y_train,
    categorical_cols,
    smoothing=20,
    n_splits=5,
    random_state=42
):
    """
    Apply out-of-fold smoothed target encoding to training data and
    train-only smoothed target encoding to validation data.
    """
    X_train_encoded = X_train.copy()
    X_val_encoded = X_val.copy()
    y_train = y_train.copy()

    encoder_artifacts = {
        "categorical_cols": categorical_cols,
        "smoothing": smoothing,
        "global_mean": float(y_train.mean()),
        "mappings": {}
    }

    skf = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state
    )

    for col in categorical_cols:
        encoded_col = f"{col}_te"
        X_train_encoded[encoded_col] = np.nan

        # Out-of-fold encoding for training rows
        for train_idx, holdout_idx in skf.split(X_train, y_train):
            X_fold_train = X_train.iloc[train_idx]
            y_fold_train = y_train.iloc[train_idx]
            X_fold_holdout = X_train.iloc[holdout_idx]

            fold_mapping, fold_global_mean = _smoothed_category_mapping(
                X_fold_train[col],
                y_fold_train,
                smoothing=smoothing
            )

            X_train_encoded.iloc[
                holdout_idx,
                X_train_encoded.columns.get_loc(encoded_col)
            ] = (
                X_fold_holdout[col]
                .astype(str)
                .map(fold_mapping)
                .fillna(fold_global_mean)
            )

        # Final mapping using full training data for validation and future inference
        full_mapping, global_mean = _smoothed_category_mapping(
            X_train[col],
            y_train,
            smoothing=smoothing
        )

        X_val_encoded[encoded_col] = (
            X_val[col]
            .astype(str)
            .map(full_mapping)
            .fillna(global_mean)
        )

        encoder_artifacts["mappings"][col] = full_mapping

    # Drop original categorical columns after encoding
    X_train_encoded = X_train_encoded.drop(columns=categorical_cols)
    X_val_encoded = X_val_encoded.drop(columns=categorical_cols)

    return X_train_encoded, X_val_encoded, encoder_artifacts


def clean_feature_names(df):
    """Clean feature names for XGBoost compatibility."""
    df = df.copy()
    df.columns = (
        df.columns
        .astype(str)
        .str.replace("[", "(", regex=False)
        .str.replace("]", ")", regex=False)
        .str.replace("<", "less_than", regex=False)
    )
    return df

## 6. Apply target encoding

In [29]:
X_train_te, X_val_te, target_encoder_artifacts = target_encode_train_validation(
    X_train=X_train,
    X_val=X_val,
    y_train=y_train,
    categorical_cols=categorical_cols,
    smoothing=20,
    n_splits=5,
    random_state=42
)

X_train_te = clean_feature_names(X_train_te)
X_val_te = clean_feature_names(X_val_te)

target_encoder_artifacts["feature_columns"] = X_train_te.columns.tolist()

print("X_train_te:", X_train_te.shape)
print("X_val_te:", X_val_te.shape)

X_train_te: (14399, 41)
X_val_te: (3600, 41)


In [30]:
# Confirm no categorical columns remain
remaining_categorical = X_train_te.select_dtypes(include=["object", "category"]).columns.tolist()
remaining_categorical

[]

In [31]:
# Confirm XGBoost-incompatible feature-name characters are gone
bad_cols = [
    col for col in X_train_te.columns
    if any(char in str(col) for char in ["[", "]", "<"])
]

bad_cols

[]

In [32]:
X_train_te.head()

,email_or_tel_available,safety_rating,high_education_ind,address_change_ind,witness_present_ind,liab_prct,policy_report_filed_ind,vehicle_made_year,vehicle_weight,vehicle_mileage,...,weekend_claim_ind,accident_site_te,channel_te,accident_type_te,in_network_bodyshop_te,liability_band_te,claim_amount_band_te,payout_vehicle_ratio_band_te,vehicle_value_band_te,accident_type_site_te
22,1,69.0,1,1,0,44.0,1,2026.0,19638.870260,65778.0,...,0,0.314820,0.228711,0.243217,0.173998,0.243750,0.232595,0.225386,0.228081,0.327867
7754,0,77.0,1,0,1,34.0,1,2022.0,8390.335837,79200.0,...,0,0.308532,0.225365,0.352192,0.171275,0.242865,0.222457,0.218685,0.223359,0.455136
11737,1,65.0,1,0,0,43.0,1,2022.0,16373.193610,101165.0,...,1,0.139676,0.229494,0.243217,0.248998,0.243750,0.217499,0.209240,0.228081,0.156658
5429,1,86.0,1,0,0,29.0,0,2023.0,21765.761130,123910.0,...,1,0.302799,0.228100,0.059447,0.166215,0.242526,0.227141,0.219198,0.224462,0.090291
14015,1,90.0,1,1,0,58.0,1,2022.0,16748.228970,89792.0,...,0,0.143064,0.228100,0.244069,0.252182,0.014387,0.223841,0.225155,0.224462,0.151191


## 7. Helper functions for evaluation and experiment logging

In [33]:
def evaluate_binary_classifier(model, X_val, y_val):
    """Evaluate a binary classifier using default threshold 0.50."""
    y_pred = model.predict(X_val)

    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_val)[:, 1]
    else:
        y_prob = y_pred

    return {
        "accuracy": round(accuracy_score(y_val, y_pred), 4),
        "precision": round(precision_score(y_val, y_pred, zero_division=0), 4),
        "recall": round(recall_score(y_val, y_pred, zero_division=0), 4),
        "f1_score": round(f1_score(y_val, y_pred, zero_division=0), 4),
        "roc_auc": round(roc_auc_score(y_val, y_prob), 4)
    }


def save_model_and_append_metrics(
    model,
    model_name,
    model_family,
    X_val,
    y_val,
    metrics_path=METRICS_PATH,
    models_dir=MODELS_DIR,
    notes=None,
    extra_artifacts=None
):
    """Save model artifact and append validation metrics to JSON experiment log."""
    timestamp = datetime.now().strftime("%Y-%m-%d_%H%M%S")
    run_id = f"{timestamp}_{model_name}"

    model_path = models_dir / f"{run_id}.pkl"

    artifact = {
        "model": model,
        "feature_columns": X_val.columns.tolist()
    }

    if extra_artifacts is not None:
        artifact.update(extra_artifacts)

    joblib.dump(artifact, model_path)

    record = {
        "run_id": run_id,
        "run_date": datetime.now().isoformat(timespec="seconds"),
        "model_name": model_name,
        "model_family": model_family,
        "model_path": str(model_path),
        "metrics": evaluate_binary_classifier(model, X_val, y_val),
        "notes": notes
    }

    if metrics_path.exists():
        with open(metrics_path, "r") as f:
            metrics_log = json.load(f)
    else:
        metrics_log = []

    metrics_log.append(record)

    with open(metrics_path, "w") as f:
        json.dump(metrics_log, f, indent=4)

    return record

## 8. Logistic Regression with target encoding

Logistic Regression benefits from scaling, so this experiment uses a `StandardScaler` inside a pipeline.

In [34]:
log_reg_te = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

log_reg_te.fit(X_train_te, y_train)

log_reg_te_record = save_model_and_append_metrics(
    model=log_reg_te,
    model_name="logistic_regression_target_encoding_v1",
    model_family="LogisticRegression",
    X_val=X_val_te,
    y_val=y_val,
    notes="Logistic regression using out-of-fold smoothed target encoding for categorical variables.",
    extra_artifacts={"target_encoder": target_encoder_artifacts}
)

log_reg_te_record["metrics"]

{'accuracy': 0.7294,
 'precision': 0.447,
 'recall': 0.774,
 'f1_score': 0.5667,
 'roc_auc': 0.8241}

In [35]:
log_reg_te_preds = log_reg_te.predict(X_val_te)

print("Confusion matrix:")
print(confusion_matrix(y_val, log_reg_te_preds))
print("Classification report:")
print(classification_report(y_val, log_reg_te_preds, zero_division=0))

Confusion matrix:
[[1989  788]
 [ 186  637]]
Classification report:
              precision    recall  f1-score   support

           0       0.91      0.72      0.80      2777
           1       0.45      0.77      0.57       823

    accuracy                           0.73      3600
   macro avg       0.68      0.75      0.69      3600
weighted avg       0.81      0.73      0.75      3600



## 9. XGBoost with target encoding

XGBoost does not need scaling. This experiment keeps the model simple so the main change being tested is the categorical encoding strategy.

In [36]:
xgb_te = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=42
)

xgb_te.fit(X_train_te, y_train)

xgb_te_record = save_model_and_append_metrics(
    model=xgb_te,
    model_name="xgboost_target_encoding_v1",
    model_family="XGBClassifier",
    X_val=X_val_te,
    y_val=y_val,
    notes="XGBoost using out-of-fold smoothed target encoding for categorical variables.",
    extra_artifacts={"target_encoder": target_encoder_artifacts}
)

xgb_te_record["metrics"]

{'accuracy': 0.8097,
 'precision': 0.6317,
 'recall': 0.4022,
 'f1_score': 0.4915,
 'roc_auc': 0.8227}

In [37]:
xgb_te_preds = xgb_te.predict(X_val_te)

print("Confusion matrix:")
print(confusion_matrix(y_val, xgb_te_preds))
print("Classification report:")
print(classification_report(y_val, xgb_te_preds, zero_division=0))

Confusion matrix:
[[2584  193]
 [ 492  331]]
Classification report:
              precision    recall  f1-score   support

           0       0.84      0.93      0.88      2777
           1       0.63      0.40      0.49       823

    accuracy                           0.81      3600
   macro avg       0.74      0.67      0.69      3600
weighted avg       0.79      0.81      0.79      3600



## 10. Compare recent model runs

This table reads the shared metrics log and compares the latest runs by model name.

In [38]:
with open(METRICS_PATH, "r") as f:
    metrics_log = json.load(f)

metrics_df = pd.DataFrame([
    {
        "run_date": record["run_date"],
        "model_name": record["model_name"],
        "model_family": record["model_family"],
        **record["metrics"]
    }
    for record in metrics_log
])

latest_metrics = (
    metrics_df
    .sort_values("run_date")
    .groupby("model_name", as_index=False)
    .tail(1)
    .sort_values("f1_score", ascending=False)
)

latest_metrics

,run_date,model_name,model_family,accuracy,precision,recall,f1_score,roc_auc
11,2026-04-28T23:30:53,logistic_regression_target_encoding_v1,LogisticRegression,0.7294,0.4470,0.7740,0.5667,0.8241
8,2026-04-28T23:07:32,logistic_regression_baseline_v2,LogisticRegression,0.7703,0.4981,0.6403,0.5603,0.8162
7,2026-04-28T23:02:40,logistic_regression_baseline_v1,LogisticRegression,0.7269,0.4430,0.7558,0.5586,0.8201
5,2026-04-28T06:17:24,xgboost_baseline_v1,XGBClassifier,0.8069,0.6103,0.4301,0.5046,0.8200
9,2026-04-28T23:07:34,xgboost_baseline_v2,XGBClassifier,0.8078,0.6180,0.4168,0.4978,0.8179
12,2026-04-28T23:30:55,xgboost_target_encoding_v1,XGBClassifier,0.8097,0.6317,0.4022,0.4915,0.8227


## 11. Notes

Interpret target encoding results against the one-hot encoded baseline:

- If F1 improves, target encoding is a useful replacement or candidate for the tuned model.
- If ROC-AUC improves but F1 does not, threshold tuning is likely the next best step.
- If both F1 and ROC-AUC decline, keep the one-hot baseline and move to threshold tuning or hyperparameter tuning.